In [1]:
%pip install pandas pdfplumber diskcache tqdm

     -------------------------------------- 60.0/60.0 kB 199.2 kB/s eta 0:00:00
     ---------------------------------------- 45.5/45.5 kB 1.1 MB/s eta 0:00:00
     ---------------------------------------- 78.4/78.4 kB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 6.6/6.6 MB 4.0 MB/s eta 0:00:00
     ---------------------------------------- 3.7/3.7 MB 10.3 MB/s eta 0:00:00
     ---------------------------------------- 3.5/3.5 MB 11.1 MB/s eta 0:00:00
     -------------------------------------- 182.8/182.8 kB 5.6 MB/s eta 0:00:00
     ---------------------------------------- 48.2/48.2 kB 2.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Extract to MS Excel with the different taxonomies

In [6]:
import os
import re
import pandas as pd
import pdfplumber
import ollama
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from hashlib import md5
from diskcache import Cache

# Caching to avoid redundant classification
cache = Cache('classification_cache')

# Load taxonomy from Excel
taxonomy_df = pd.read_excel("library/taxonomy.xlsx")
taxonomy = [
    {"Category": row["Category"], "Description": row["Description"]}
    for _, row in taxonomy_df.iterrows()
]

# System prompt with full taxonomy description
taxonomy_prompt = "\n".join([f"- {t['Category']}: {t['Description']}" for t in taxonomy])
system_prompt = (
    "You are an assistant helping classify evaluation content into child protection sub-categories.\n\n"
    "Only classify paragraphs that provide an evaluation (judgement, evidence, assessment, or conclusion) for a category.\n"
    "Return a comma-separated list of all relevant category names. Return 'None' if not relevant to any.\n"
    f"Taxonomy:\n{taxonomy_prompt}"
)

# Classification function with caching
def classify_paragraph(paragraph):
    key = md5(paragraph.encode()).hexdigest()
    if key in cache:
        return cache[key]

    try:
        response = ollama.chat(
            model='mistral',
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"""Evaluate if the following paragraph includes evaluative content (judgement, evidence, assessment, or conclusion) about any child protection sub-category listed.\n\nReturn a comma-separated list of matching category names. Return 'None' if not relevant.\n\nParagraph:\n\"\"\"{paragraph}\"\"\"\n\nCategories:"""}
            ]
        )
        category_list = response['message']['content'].strip()
        cache[key] = category_list
        return category_list
    except Exception as e:
        print(f"Error classifying paragraph: {e}")
        return "None"

# Function to sanitize paragraph by removing hard returns
def clean_paragraph(text):
    return re.sub(r'\s*\n\s*', ' ', text).strip()

# Worker function for threaded processing
def process_paragraph(report_name, page_num, paragraph):
    clean_text = clean_paragraph(paragraph)
    categories_str = classify_paragraph(clean_text)
    if categories_str.lower() == "none":
        return []
    
    categories = [cat.strip() for cat in categories_str.split(",") if cat.strip()]
    return [{
        "File name": report_name,
        "Category": category,
        "Extracted paragraph (verbatim)": clean_text,
        "¶ / page / line reference": f"[p.{page_num}]"
    } for category in categories]

# Split text into paragraphs of ~5 sentences
def split_text_to_paragraphs(text, max_sentences=5):
    sentence_endings = re.compile(r'(?<=[.!?])\s+')
    sentences = sentence_endings.split(text.strip())
    
    paragraphs = []
    for i in range(0, len(sentences), max_sentences):
        chunk = sentences[i:i+max_sentences]
        paragraph = " ".join(chunk).strip()
        if len(paragraph) > 30:
            paragraphs.append(paragraph)
    return paragraphs

import multiprocessing
num_workers = multiprocessing.cpu_count()
# MAIN SCRIPT
report_path = "reports/test_short.pdf"
report_name = os.path.basename(report_path)
output_rows = []

with pdfplumber.open(report_path) as pdf:
    all_tasks = []
    with ThreadPoolExecutor(max_workers=32) as executor:
        for i, page in enumerate(tqdm(pdf.pages, desc="Reading PDF")):
            text = page.extract_text()
            if not text:
                continue
            paragraphs = split_text_to_paragraphs(text)
            for paragraph in paragraphs:
                all_tasks.append(executor.submit(process_paragraph, report_name, i + 1, paragraph))

        for future in tqdm(as_completed(all_tasks), total=len(all_tasks), desc="Classifying Paragraphs"):
            results = future.result()
            if results:
                output_rows.extend(results)

# Convert to Markdown table format
def to_markdown_table(rows):
    lines = ["| File name | Category | Extracted paragraph (verbatim) | ¶ / page / line reference |",
             "|-----------|----------|--------------------------------|----------------------------|"]
    for row in rows:
        lines.append(f"| {row['File name']} | {row['Category']} | {row['Extracted paragraph (verbatim)'].replace('|', '\\|')} | {row['¶ / page / line reference']} |")
    return "\n".join(lines)

# Save output
df_out = pd.DataFrame(output_rows)
df_out.to_excel("docs/classified_output_multicategory.xlsx", index=False)
with open("docs/classified_output_multicategory.md", "w", encoding="utf-8") as f:
    f.write(to_markdown_table(output_rows))

print("Done. Output saved to 'docs/classified_output_multicategory.xlsx' and 'docs/classified_output_multicategory.md'")


Classifying Paragraphs: 100%|██████████| 15/15 [00:00<?, ?it/s]

Done. Output saved to 'docs/classified_output_multicategory.xlsx' and 'docs/classified_output_multicategory.md'


## Now generate a Summary Report

In [ ]:
import docx
from docx.shared import Pt
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT

# Summarization function
def generate_summary(df_out):
    # Group paragraphs by category
    grouped = df_out.groupby("Category")["Extracted paragraph (verbatim)"].apply(list).to_dict()

    # Build the summarization prompt
    category_sections = []
    for category, paragraphs in grouped.items():
        joined_text = "\n".join(paragraphs)
        category_sections.append(f"Category: {category}\n\n{joined_text}\n")

    user_prompt = (
        "Generate a structured summary report from the following classified evaluation content.\n\n"
        + "\n\n".join(category_sections)
        + "\n\nFollow the instructions provided."
    )

    response = ollama.chat(
        model="mistral",
        messages=[
            {"role": "system", "content": (
                            "You are an assistant helping to generate a structured summary report from classified evaluation content.\n\n"
            "Instructions:\n"
            "- Begin with an **Introduction** that briefly explains that you are generating a summary report from an evaluation report.\n"
            "- Group extracted paragraphs by their assigned Category.\n"
            "- For each Category:\n"
            "  - Provide a concise summary of the evaluative content (judgement, evidence, assessment, or conclusion).\n"
            "  - Highlight recurring themes, strengths, weaknesses, and recommendations if present.\n"
            "  - Avoid repeating verbatim text unless necessary for clarity.\n"
            "- End with a **Conclusion** that synthesizes findings across all categories, noting cross-cutting issues or general conclusions.\n"
            "- Keep the tone professional, neutral, and analytical."
            )},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response['message']['content'].strip()

# Generate summary text
summary_text = generate_summary(df_out)

# Create Word document
doc = docx.Document()

# Title
title = doc.add_heading("Child Protection Evaluation Summary Report", level=0)
title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

# Add summary text
for line in summary_text.split("\n"):
    para = doc.add_paragraph(line)
    para.style.font.size = Pt(11)

# Save Word file
doc.save("docs/summary_report.docx")

print("Summary report saved to 'docs/summary_report.docx'")


Summary report saved to 'summary_report.docx'
